<p>
  <img src="../assets/latincy-ext-logo.jpg" alt="LatinCy Ext" width="380">
</p>

# Macron Morph — demo

A **non-destructive** component that uses macron (long-vowel) marks to disambiguate Latin
morphology. It reads a macronized token form, looks it up in a kaikki-derived table, and sets
`token._.macron_morph` / `token._.macron_pos_` — alternative annotations that never override
the model's own predictions.

This notebook walks through the component end to end:

1. **Live disambiguation** — the ablative `puellā`, resolved deterministically from its macron
2. **`venit` vs. `vēnit`** — the canonical present/perfect ambiguity, identical without macrons
3. **Partial disambiguation** — an ambiguous form where the lookup confirms `Number`/`pos` but not `Case`

---

> **Data.** The lookup table (`latin-forms-macronized-morph.json`) is kaikki-derived (harvested
> from Wiktionary) and is **not bundled** with this package. It is built by
> `extract_macronized_morph.py` in [latincy-words](https://github.com/diyclassics/latincy-words).

## Setup

The component is packaged as `latincy_ext`. The macron lookup table lives in the
`latincy-words` repo (not bundled here) — build it there, then point the component at it via
the `lookup_path` config or the `LATINCY_MACRON_LOOKUP` environment variable. The `show()`
helper below prints the model's morphology alongside the macron signal for comparison.

In [ ]:
import os
import spacy
import latincy_ext  # registers the macron_morph @Language.factory
from pathlib import Path

# The kaikki-derived lookup lives in latincy-words. Override with LATINCY_MACRON_LOOKUP.
LOOKUP = Path(os.environ.get(
    "LATINCY_MACRON_LOOKUP",
    "../../latincy-words/data/processed/latin-forms-macronized-morph.json",
)).resolve()
assert LOOKUP.exists(), (
    f"lookup not found: {LOOKUP}\n"
    "  build it: cd latincy-words && uv run python scripts/extract_macronized_morph.py"
)

nlp = spacy.load("la_core_web_lg")
nlp.add_pipe("macron_morph", last=True, config={"lookup_path": str(LOOKUP)})
print("pipeline:", nlp.pipe_names)


def show(doc):
    """Print each token's model morphology next to the macron-derived signal."""
    header = f"{'TOKEN':<14} {'POS':<7} {'MODEL MORPH':<50} {'MACRON MORPH':<40} {'MACRON POS'}"
    print(header)
    print("-" * len(header))
    for token in doc:
        print(
            f"{token.text:<14} "
            f"{token.pos_:<7} "
            f"{str(token.morph):<50} "
            f"{token._.macron_morph:<40} "
            f"{token._.macron_pos_}"
        )

## 1 · Live disambiguation — ablative of agent

`puellā` is ablative singular. Without the macron the model must infer this from context;
with the macron the lookup is deterministic.

In [ ]:
doc = nlp("A puellā res facta est.")
show(doc)

## 2 · `venit` vs. `vēnit` — present vs. perfect

The canonical ambiguity: `venit` (present, *she comes*) vs. `vēnit` (perfect, *she came*).
Both surface identically without macrons; the macron resolves the tense.

In [ ]:
present = nlp("Nunc venit puella.")
perfect = nlp("Heri vēnit puella.")

print("Present (venit):")
show(present)
print()
print("Perfect (vēnit):")
show(perfect)

## 3 · Partial disambiguation — ambiguous form

`thēsaurō` is both dative and ablative singular. The lookup cannot resolve `Case`, but it can
still confirm `Number=Sing` and `pos=NOUN` — the intersection of all matching parses.

In [ ]:
doc = nlp("Thēsaurō gaudet.")
show(doc)

## Takeaways

- **Non-destructive by design.** The component sets `token._.macron_morph` / `token._.macron_pos_`
  only — it never touches `token.morph` or `token.pos_`.
- **Empty is falsy.** `token._.macron_morph` is an empty string when no macrons are present or the
  form isn't in the lookup — safe to use as a truthiness check.
- **Intersection semantics.** For ambiguous forms, only features agreed across **all** matching
  parses are set (see `thēsaurō` above).
- **Plain text?** To use with unmacronized input, add the `macronizer` component (from
  `latincy-macronizer`) before `macron_morph`:

```python
nlp.add_pipe("macronizer", config={"model_path": "..."})   # from latincy-macronizer
nlp.add_pipe("macron_morph", last=True, config={"lookup_path": "..."})
```